In [1]:
import json
import pandas as pd

versicode_edit = json.load(
    open("../benchmark/VersiCode_Benchmark/code_editing/code_editing_old_to_new.json")
)["data"]
versicode_edit = pd.DataFrame(versicode_edit)
print(len(versicode_edit))

49346


In [2]:
versicode_edit["dependency"].unique()

array(['click', 'datasets', 'imageio', 'jax', 'jedi', 'keras', 'librosa',
       'MarkupSafe', 'paddlepaddle-gpu', 'pandas', 'pyrsistent',
       'pytorch-lightning', 'ray', 'streamlit', 'tensorflow', 'torch'],
      dtype=object)

In [3]:
versicode_edit[["dependency", "old_name", "new_name"]].drop_duplicates().to_csv(
    "../benchmark/VersiCode_Benchmark/code_editing/Python_API_pairs.csv", index=False
)

In [4]:
repfinder_data = json.load(open("../benchmark/RepFinder/groundtruth.json"))
records = []
for library, API_changes in repfinder_data["Y"].items():
    group_id, artifact_id = library.split("__XX__")[:2]
    for version_pair, data in API_changes.items():
        version_before, version_after = version_pair.split("__XX__", 1)
        for api_before, api_after in data.items():
            return_type, api_sig = api_before.split("__XX__", 1)
            tmp = {
                "name": f"{group_id}:{artifact_id}",
                "version_before": version_before,
                "version_after": version_after,
                "return_type": return_type,
                "api_before": api_sig,
                "api_after": api_after["repl_method"],
                "map_type": api_after["n_to_n"],
                "source": api_after["source"],
                "relation": api_after["relation"],
            }
            records.append(tmp)

repfinder_data = pd.DataFrame(records)
print(len(repfinder_data))

583


In [5]:
repfinder_data["map_type"].value_counts()

map_type
1 to 1    528
1 to n     26
n to n     16
1 to i      9
n to 1      4
Name: count, dtype: int64

In [6]:
repfinder_data_1to1 = repfinder_data[repfinder_data["map_type"] == "1 to 1"].copy()
repfinder_data_1to1.loc[:, "api_after"] = repfinder_data_1to1["api_after"].map(
    lambda x: x[0]
)
repfinder_data_1to1.to_csv("../benchmark/RepFinder/Java_API_pairs.csv", index=False)

In [7]:
repfinder_data_1to1["api_before"].value_counts()

api_before
org.elasticsearch.common.settings.ImmutableSettings.settingsBuilder()            7
org.elasticsearch.action.bulk.BulkItemResponse.failed()                          6
org.apache.lucene.queryParser.QueryParser.parse(String)                          5
org.elasticsearch.action.search.SearchRequestBuilder.setFilter(FilterBuilder)    5
org.elasticsearch.common.settings.Settings.settingsBuilder()                     5
                                                                                ..
com.tencent.angel.ml.core.conf.MLConf.ML_MODEL_CLASS_NAME()                      1
com.tencent.angel.ml.core.conf.MLConf.ML_GBDT_TREE_DEPTH()                       1
com.tencent.angel.ml.core.conf.MLConf.ML_GBDT_TREE_NUM()                         1
com.tencent.angel.ml.core.conf.MLConf.ML_NUM_CLASS()                             1
com.tencent.angel.ml.core.conf.MLConf.ML_RANK_NUM()                              1
Name: count, Length: 324, dtype: int64

In [8]:
repfinder_data_1to1[
    (repfinder_data_1to1["relation"] == "Pull Up Method")
    & (repfinder_data_1to1["source"] != "External Library")
]

,name,version_before,version_after,return_type,api_before,api_after,map_type,source,relation
0,org.mockito:mockito-core,2.0.33-beta,2.2.17,java.lang.String,org.mockito.Matchers.matches(String),org.mockito.ArgumentMatchers.matches(String),1 to 1,Deprecation Message,Pull Up Method
1,org.mockito:mockito-core,2.0.33-beta,2.2.17,java.lang.String,org.mockito.Matchers.anyString(),org.mockito.ArgumentMatchers.anyString(),1 to 1,Deprecation Message,Pull Up Method
2,org.mockito:mockito-core,1.9.5,3.5.10,java.lang.String,org.mockito.Matchers.anyString(),org.mockito.ArgumentMatchers.anyString(),1 to 1,Deprecation Message,Pull Up Method
3,org.mockito:mockito-core,1.9.5,3.5.10,java.lang.Object,org.mockito.Matchers.any(),org.mockito.ArgumentMatchers.any(),1 to 1,Deprecation Message,Pull Up Method
4,org.mockito:mockito-core,1.9.5,3.5.10,java.lang.Object,org.mockito.Matchers.anyObject(),org.mockito.ArgumentMatchers.anyObject(),1 to 1,Deprecation Message,Pull Up Method
...,...,...,...,...,...,...,...,...,...
441,org.rocksdb:rocksdbjni,3.10.1,5.11.3,org.rocksdb.Options,org.rocksdb.Options.setRateLimiterConfig(RateL...,org.rocksdb.Options.setRateLimiter(RateLimiter),1 to 1,Own Library,Pull Up Method
553,org.jmock:jmock,2.5.1,2.8.3,void,org.jmock.Expectations.will(Action),org.jmock.AbstractExpectations.will(Action),1 to 1,Own Library,Pull Up Method
554,org.jmock:jmock,2.5.1,2.8.3,org.jmock.api.Action,org.jmock.Expectations.returnValue(Object),org.jmock.AbstractExpectations.returnValue(Obj...,1 to 1,Own Library,Pull Up Method
555,org.jmock:jmock,2.5.1,2.8.3,org.jmock.api.Action,org.jmock.Expectations.throwException(Throwable),org.jmock.AbstractExpectations.throwException(...,1 to 1,Own Library,Pull Up Method


In [12]:
name_cases = {"identical": 0, "substring": 0, "substring_r": 0}
for row in repfinder_data_1to1.itertuples(index=False):
    name_before = row.api_before.split("(", 1)[0].split(".")[-1].lower()
    name_after = row.api_after.split("(", 1)[0].split(".")[-1].lower()
    if name_before == name_after:
        name_cases["identical"] += 1
    elif name_before in name_after:
        name_cases["substring"] += 1
    elif name_after in name_before:
        name_cases["substring_r"] + 1
    else:
        print(row.api_before, row.api_after)

org.apache.lucene.document.Document.getFieldables(String) org.apache.lucene.document.Document.getFields(String)
org.apache.lucene.index.IndexWriter.optimize() org.apache.lucene.index.IndexWriter.forceMerge(int)
org.apache.lucene.store.NoLockFactory.getNoLockFactory() org.apache.lucene.store.NoLockFactory.INSTANCE
org.apache.lucene.index.IndexWriter.getCommitData() org.apache.lucene.index.IndexWriter.getLiveCommitData()
org.apache.lucene.index.IndexWriter.optimize() org.apache.lucene.index.IndexWriter.forceMerge(int)
org.elasticsearch.search.aggregations.metrics.tophits.TopHitsBuilder.addField(String) org.elasticsearch.search.aggregations.metrics.tophits.TopHitsAggregationBuilder.storedField(String)
org.elasticsearch.action.search.SearchRequestBuilder.setFilter(FilterBuilder) org.elasticsearch.action.search.SearchRequestBuilder.setPostFilter(QueryBuilder)
org.elasticsearch.action.delete.DeleteResponse.isNotFound() org.elasticsearch.action.delete.DeleteResponse.status()
org.elasticsearch

In [13]:
name_cases

{'identical': 372, 'substring': 43, 'substring_r': 0}